# Question Relevance Evaluation

First install necessary libaries and setup OpenAI key.

In [1]:
!pip install openai

In [2]:
from openai import OpenAI
import ast
import csv
import pandas as pd
import time
import random

In [ ]:
client = OpenAI(api_key="sk-proj-")

## Evaluator

First we create the evalutor.

In [4]:
evaluator_prompt = """
You are an evaluation agent.

Given:
1. A block of slide text.
2. A generated question.

Your task is to evaluate whether the question is relevant to the slide text.

Rules:
- "relevant" = The question is directly grounded in the slide content and supported by information in the text.
- "irrelevant" = The question is not grounded in the slide content, is only loosely related, or lacks meaningful support from the text.

Return ONLY one word: relevant or irrelevant.
"""

Define a function that will predict the score using the evaluator.

In [5]:
def get_relevance_label(text, question):
    user_content = f"Slide text:\n{text}\n\nQuestion:\n{question}\n\nReturn relevance label only."
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": evaluator_prompt},
            {"role": "user", "content": user_content}
        ],
        temperature=0
    )

    label = response.choices[0].message.content.strip().lower()
    return label

Evalaute our self labeled dataset.

In [6]:
df = pd.read_csv("text_questions_relevance.csv")

model_labels = []
correct_flags = []

for i, row in df.iterrows():
    text = row["text"]
    question = row["question"]

    # Convert CSV label ("yes"/"no") → ("relevant"/"irrelevant")
    raw = row["relevant"].strip().lower()

    if raw == "yes":
        true_label = "relevant"
    else:
        true_label = "irrelevant"

    # 1. Model-generated label
    model_label = get_relevance_label(text, question)
    model_labels.append(model_label)

    # 2. Compare model vs true
    correct = (model_label == true_label)
    correct_flags.append(correct)


Print the accuracy

In [7]:
accuracy = sum(correct_flags) / len(correct_flags)
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 1.0000


## Evaluate Question Generator

Set the prompt for generated questions

In [ ]:
system_prompt_three_level = """
You are an agent that generates questions at a specified Bloom’s Taxonomy level.
You will receive two inputs:
1. Raw text extracted from lecture slides.
2. A level request: "remember", "understand", or "apply".

Your task:
Create ONE question at the requested level based only on the information in the slide text.

Rules by level:

remember:
- Ask for simple factual recall only.
- The question must require recalling a fact, term, definition, list, or identification.
- One clear factual answer.
- No explanation, interpretation, mechanism, reasoning, or relationships.
- Do NOT ask what determines, causes, influences, controls, affects, or results in anything.
- Do NOT ask about functions or purposes unless they are explicitly stated facts.
- Do NOT ask about properties unless the property itself is a memorized fact.
- No scenario, no prediction, no cause–effect language.

understand:
- Ask the learner to explain, describe, summarize, or interpret a concept.
- No scenario-based application or problem-solving.
- The question should test comprehension of meaning, relationships, or mechanisms.
- Do NOT ask the learner to predict outcomes in new situations.

apply:
- Create a NEW scenario, event, or condition that is directly relevant to the information.
- The scenario must include a change, malfunction, variation, or specific situation the learner must reason about.
- The learner must USE information from the text to PREDICT an outcome, determine a result, or identify the consequence of that change.
- The answer must be a single, logically deducible outcome.
- The question must be an ACTUAL question ending with a question mark.
- Do NOT reveal, imply, or restate the outcome inside the question.
- Do NOT write the question as a statement or give away the result (e.g., “If X happens, the tissue would…”).
- Do NOT ask for interpretation of signs, meaning, or function (“what does this indicate,” “what does this mean”).
- Do NOT use explanation-style wording (“why,” “explain,” “describe”).
- Do NOT ask the learner to restate normal function; require a predicted outcome of the scenario.

General rules:
- Do NOT reference “the slide” or “the text.”
- Use only information found in the input text.
- Keep the question clear, concise, and natural.
- If the slide text is sparse, choose any fact present and build the question around it.
- The output must ALWAYS be a question.

Output:
Only the question.


"""

Generate questions froms slides

In [ ]:
def generate_questions_from_slides(
    input_csv="slides_extracted.csv",
    system_prompt="",
    output_csv="questions_output.csv",
    model="gpt-4.1",
    questions_per_slide=20
):
    bloom_labels = ["remember", "understand", "apply"]
    rows = []

    # Load slide text
    with open(input_csv, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            rows.append(row["text"])

    # Open output file
    with open(output_csv, "w", newline="") as f_out:
        writer = csv.writer(f_out)
        writer.writerow(["question", "slide_text", "bloom_label"])

        for slide_index, slide_text in enumerate(rows):
            for j in range(questions_per_slide):

                # Pick random Bloom label
                label = random.choice(bloom_labels)

                # Build combined prompt
                user_prompt = f"Bloom level: {label}\n\nSlide text:\n{slide_text}"

                time.sleep(1.1)

                # Query model
                response = client.chat.completions.create(
                    model=model,
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt}
                    ]
                )

                question = response.choices[0].message.content.strip()

                # Write to CSV
                writer.writerow([question, slide_text, label])

                print(f"{slide_index+1}-{j+1}: [{label}] {question}")

    print(f"\nSaved {len(rows) * questions_per_slide} questions to {output_csv}")


In [ ]:
generate_questions_from_slides(
    system_prompt=system_prompt_three_level,
)

1-1: [remember] What are the names of the lobes that make up the cerebral hemispheres?
1-2: [understand] How is the brain organized into its major subdivisions and what structures are included in each subdivision?
1-3: [remember] What fissure separates the right and left cerebral hemispheres?
1-4: [remember] What are the names of the four lobes of the cerebral hemispheres?
1-5: [remember] What are the names of the two fissures mentioned that separate major parts of the brain?
1-6: [apply] If a blockage developed in the transverse fissure separating the cerebrum and cerebellum, which two brain regions would have their separation impacted?
1-7: [apply] If a coronal MRI shows a mass located within the longitudinal fissure, which anatomical region would be affected?
1-8: [understand] Describe how the transverse fissure and the longitudinal fissure separate different parts of the brain.
1-9: [remember] What is the name of the fissure between the right and left cerebral hemispheres?
1-10: [r

Now run the evalutor on the generated question, text pairs.

In [10]:
def run_binary_evaluator(
    input_csv="text_questions.csv",
    model="gpt-4.1-mini"
):
    texts = []
    questions = []

    with open(input_csv, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            texts.append(row["slide_text"])
            questions.append(row["question"])

    labels = []

    for slide, q in zip(texts, questions):
        time.sleep(1.1)

        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": evaluator_prompt},
                {"role": "user", "content": f"Slide text:\n{slide}\n\nQuestion:\n{q}"}
            ]
        )

        label = response.choices[0].message.content.strip().lower()
        if label in ["relevant", "irrelevant"]:
            labels.append(label)

    relevance_rate = sum(l == "relevant" for l in labels) / len(labels)
    print(f"Relevance rate: {relevance_rate:.4f}")

    return relevance_rate


In [11]:
relevance_rate = run_binary_evaluator(
    input_csv="text_questions.csv",
    model="gpt-4.1-mini"
)

Relevance rate: 0.9950


TypeError: cannot unpack non-iterable float object

Now run the same evaluation but this time seperate by bloom level.

In [15]:
def run_binary_evaluator_loop_by_bloom(
    input_csv="questions_output.csv",
    model="gpt-4.1-mini"
):
    texts = []
    questions = []
    blooms = []

    with open(input_csv, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            texts.append(row["slide_text"])
            questions.append(row["question"])
            blooms.append(row["bloom_label"].strip().lower())

    bloom_counts = {
        "remember": [],
        "understand": [],
        "apply": []
    }

    for slide, q, bloom in zip(texts, questions, blooms):
        time.sleep(1.1)

        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": evaluator_prompt},
                {"role": "user", "content": f"Slide text:\n{slide}\n\nQuestion:\n{q}"}
            ]
        )

        label = response.choices[0].message.content.strip().lower()

        if label in ["relevant", "irrelevant"]:
            bloom_counts[bloom].append(label)

    # Compute relevance rate per Bloom level
    bloom_relevance_rate = {
        bloom: (
            sum(label == "relevant" for label in labels) / len(labels)
            if len(labels) > 0 else 0.0
        )
        for bloom, labels in bloom_counts.items()
    }

    print("Relevance rate by Bloom level:")
    for bloom in ["remember", "understand", "apply"]:
        print(f"{bloom}: {bloom_relevance_rate[bloom]:.4f}")

    return bloom_relevance_rate

In [17]:
scores, avg_score, avg_by_bloom = run_binary_evaluator_loop_by_bloom(
    input_csv="text_questions.csv",
    model="gpt-4.1-mini"
)

Relevance rate by Bloom level:
remember: 1.0000
understand: 1.0000
apply: 0.9685


In [ ]:
avg_by_bloom

{'remember': 96.15384615384616,
 'understand': 89.3006993006993,
 'apply': 84.94029850746269}